# Problem Statement

The objective of this project is to identify which customers are likely to make a transaction in the future.

The dataset contains anonymized customer information with **200 features**, along with an `ID_code` and a binary `target` variable:

* `0` → Customer will not make a transaction
* `1` → Customer will make a transaction

The main goal is to build and compare multiple machine learning classification models and identify a suitable model for predicting future customer transactions.

Since the features are anonymized, the project focuses on data analysis, preprocessing, model development, and predictive performance rather than interpreting individual features.


# Import Required Libraries

In [1]:
import warnings

import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

## Load the data

In [2]:
df = pd.read_csv("train(1).csv")

In [3]:
df.head()

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


## Basic checks

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Columns: 202 entries, ID_code to var_199
dtypes: float64(200), int64(1), object(1)
memory usage: 308.2+ MB


In [5]:
df.columns

Index(['ID_code', 'target', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4',
       'var_5', 'var_6', 'var_7',
       ...
       'var_190', 'var_191', 'var_192', 'var_193', 'var_194', 'var_195',
       'var_196', 'var_197', 'var_198', 'var_199'],
      dtype='object', length=202)

In [6]:
df['target'].value_counts() # checking target column

target
0    179902
1     20098
Name: count, dtype: int64

In [7]:
df.isnull().sum().sum() # checking null values

np.int64(0)

In [8]:
df.dtypes.value_counts() # checking the data types

float64    200
object       1
int64        1
Name: count, dtype: int64

In [9]:
df['ID_code'].head()

0    train_0
1    train_1
2    train_2
3    train_3
4    train_4
Name: ID_code, dtype: object

In [10]:
df = df.drop("ID_code", axis=1)  # ID_code is only an identifier, so it is removed

In [11]:
df.duplicated().sum()  # checking duplicate rows (after removing ID_code)

np.int64(0)

In [12]:
df.head()

,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,-4.9200,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,3.1468,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,-4.9193,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,-5.8609,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,6.2654,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [13]:
df.describe()

,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
count,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,...,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000
mean,0.100490,10.679914,-1.627622,10.715192,6.796529,11.078333,-5.065317,5.408949,16.545850,0.284162,...,3.234440,7.438408,1.927839,3.331774,17.993784,-0.142088,2.303335,8.908158,15.870720,-3.326537
std,0.300653,3.040051,4.050044,2.640894,2.043319,1.623150,7.863267,0.866607,3.418076,3.332634,...,4.559922,3.023272,1.478423,3.992030,3.135162,1.429372,5.454369,0.921625,3.010945,10.438015
min,0.000000,0.408400,-15.043400,2.117100,-0.040200,5.074800,-32.562600,2.347300,5.349700,-10.505500,...,-14.093300,-2.691700,-3.814500,-11.783400,8.694400,-5.261000,-14.209600,5.960600,6.299300,-38.852800
25%,0.000000,8.453850,-4.740025,8.722475,5.254075,9.883175,-11.200350,4.767700,13.943800,-2.317800,...,-0.058825,5.157400,0.889775,0.584600,15.629800,-1.170700,-1.946925,8.252800,13.829700,-11.208475
50%,0.000000,10.524750,-1.608050,10.580000,6.825000,11.108250,-4.833150,5.385100,16.456800,0.393700,...,3.203600,7.347750,1.901300,3.396350,17.957950,-0.172700,2.408900,8.888200,15.934050,-2.819550
75%,0.000000,12.758200,1.358625,12.516700,8.324100,12.261125,0.924800,6.003000,19.102900,2.937900,...,6.406200,9.512525,2.949500,6.205800,20.396525,0.829600,6.556725,9.593300,18.064725,4.836800
max,1.000000,20.315000,10.376800,19.353000,13.188300,16.671400,17.251600,8.447700,27.691800,10.151300,...,18.440900,16.716500,8.402400,18.281800,27.928800,4.272900,18.321500,12.000400,26.079100,28.500700


# Define X and Y 

In [14]:
x = df.drop('target',axis=1)
y = df['target']

In [15]:
print(x.shape)
print(y.shape)

(200000, 200)
(200000,)


In [16]:
Q1 = x.quantile(0.25)
Q3 = x.quantile(0.75)

IQR = Q3 - Q1

outliers = ((x < (Q1 - 1.5 * IQR)) | (x > (Q3 + 1.5 * IQR))).sum()

outliers.sort_values(ascending=False).head(10)


var_179    1509
var_44     1208
var_93      889
var_146     804
var_192     733
var_31      637
var_133     589
var_105     572
var_38      502
var_124     498
dtype: int64

**Note on outliers:** the IQR check above is only for inspection. No rows were removed, because the features are anonymized and the models used are not strongly affected by a small share of extreme values.

## Data Analysis Summary

* **Size and types:** 200,000 rows; 200 numeric features (`var_0`–`var_199`, all `float64`), plus `ID_code` (text identifier) and `target` (integer).
* **Target balance:** 179,902 customers with `target = 0` and 20,098 with `target = 1`, so only about **10%** of customers make a transaction. The data is highly imbalanced.
* **Missing values:** none found across the dataset.
* **Duplicates:** checked on the feature and target columns after removing `ID_code`, since `ID_code` is unique for every row.
* **Outliers:** the IQR check shows some features contain many extreme values (for example `var_179` has 1,509 and `var_44` has 1,208). They were inspected but not removed.
* **Feature interpretation:** features are anonymized, so as the project brief allows, detailed EDA on individual features was skipped and the focus is on modelling.


# Train test split

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state=42,stratify=y)

In [18]:
x_train.shape

(160000, 200)

In [19]:
y_train.value_counts()

target
0    143922
1     16078
Name: count, dtype: int64

In [20]:
y_test.value_counts()

target
0    35980
1     4020
Name: count, dtype: int64

# Balancing training data

In [21]:
# balancing the training data while keeping the test data in original distribution for realistic evaluation
rus = RandomUnderSampler(random_state=42)

x_train_balanced, y_train_balanced = rus.fit_resample(
    x_train, y_train
)

y_train_balanced.value_counts()

target
0    16078
1    16078
Name: count, dtype: int64

# Scaling

In [22]:
scaler = StandardScaler()
x_train_balanced_scaled = scaler.fit_transform(x_train_balanced)
x_test_scaled = scaler.transform(x_test)

In [23]:
# Model 1: Logistic Regression
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(x_train_balanced_scaled, y_train_balanced)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [24]:
y_pred_lr = model_lr.predict(x_test_scaled)

In [25]:
y_pred_lr

array([0, 0, 0, ..., 0, 1, 0], shape=(40000,))

In [26]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

y_prob_lr = model_lr.predict_proba(x_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, y_prob_lr)

print("Logistic Regression ROC-AUC:", auc_lr)

Accuracy: 0.78265
              precision    recall  f1-score   support

           0       0.97      0.78      0.87     35980
           1       0.29      0.77      0.42      4020

    accuracy                           0.78     40000
   macro avg       0.63      0.78      0.64     40000
weighted avg       0.90      0.78      0.82     40000

Logistic Regression ROC-AUC: 0.8583859537775271


In [27]:
# Model 2: Random Forest
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(x_train_balanced, y_train_balanced)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
y_pred_rf = model_rf.predict(x_test)

In [29]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

y_prob_rf = model_rf.predict_proba(x_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest ROC-AUC:", auc_rf)

Accuracy: 0.74935
              precision    recall  f1-score   support

           0       0.96      0.75      0.84     35980
           1       0.24      0.70      0.36      4020

    accuracy                           0.75     40000
   macro avg       0.60      0.73      0.60     40000
weighted avg       0.89      0.75      0.80     40000

Random Forest ROC-AUC: 0.8049826741777494


In [30]:
# Model 3: XGBoost

model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

model_xgb.fit(x_train_balanced, y_train_balanced)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [31]:
y_pred_xgb = model_xgb.predict(x_test)

In [32]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

y_prob_xgb = model_xgb.predict_proba(x_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print("XGBoost ROC-AUC:", auc_xgb)

Accuracy: 0.782425
              precision    recall  f1-score   support

           0       0.96      0.79      0.87     35980
           1       0.28      0.72      0.40      4020

    accuracy                           0.78     40000
   macro avg       0.62      0.76      0.63     40000
weighted avg       0.89      0.78      0.82     40000

XGBoost ROC-AUC: 0.8365585289229228


In [33]:
# Model 4: Support Vector Machine (Linear SVM)

model_svm = LinearSVC(
    random_state=42,
    max_iter=2000
)

model_svm.fit(x_train_balanced_scaled, y_train_balanced)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [34]:
y_pred_svm = model_svm.predict(x_test_scaled)

In [35]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

y_score_svm = model_svm.decision_function(x_test_scaled)

auc_svm = roc_auc_score(y_test, y_score_svm)

print("Linear SVM ROC-AUC:", auc_svm)

Accuracy: 0.78285
              precision    recall  f1-score   support

           0       0.97      0.78      0.87     35980
           1       0.29      0.77      0.42      4020

    accuracy                           0.78     40000
   macro avg       0.63      0.78      0.64     40000
weighted avg       0.90      0.78      0.82     40000

Linear SVM ROC-AUC: 0.8583384425841887


In [36]:
# Model 5: Decision Tree

model_dt = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

model_dt.fit(x_train_balanced, y_train_balanced)

,criterion,'gini'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [37]:
y_pred_dt = model_dt.predict(x_test)

In [38]:
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

y_prob_dt = model_dt.predict_proba(x_test)[:, 1]

auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree ROC-AUC:", auc_dt)

Accuracy: 0.7084
              precision    recall  f1-score   support

           0       0.93      0.73      0.82     35980
           1       0.17      0.47      0.25      4020

    accuracy                           0.71     40000
   macro avg       0.55      0.60      0.53     40000
weighted avg       0.85      0.71      0.76     40000

Decision Tree ROC-AUC: 0.6282482459851935


Decision Tree ROC-AUC: 0.6282482459851935


In [39]:
# Model 6: Gradient Boosting

model_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_gb.fit(x_train_balanced, y_train_balanced)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [40]:
y_pred_gb = model_gb.predict(x_test)

In [41]:
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

y_prob_gb = model_gb.predict_proba(x_test)[:, 1]
auc_gb = roc_auc_score(y_test, y_prob_gb)

print("GradientBoosting ROC-AUC:", auc_gb)

Accuracy: 0.772625
              precision    recall  f1-score   support

           0       0.96      0.78      0.86     35980
           1       0.26      0.71      0.38      4020

    accuracy                           0.77     40000
   macro avg       0.61      0.74      0.62     40000
weighted avg       0.89      0.77      0.81     40000

GradientBoosting ROC-AUC: 0.8269465761796908


In [42]:
# Confusion matrix for the selected model (Logistic Regression)
cm = confusion_matrix(y_test, y_pred_lr)

print(cm)

[[28199  7781]
 [  913  3107]]


# Model Comparison

Multiple classification models were trained and evaluated on the customer transaction prediction problem.

The following metrics were considered:

* Accuracy
* Precision
* Recall
* F1-score
* ROC-AUC

| Model | Accuracy | Class 1 Precision | Class 1 Recall | Class 1 F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.7827 | 0.29 | 0.77 | 0.42 | 0.8584 |
| Random Forest | 0.7494 | 0.24 | 0.70 | 0.36 | 0.8050 |
| XGBoost | 0.7824 | 0.28 | 0.72 | 0.40 | 0.8366 |
| Linear SVM | 0.7829 | 0.29 | 0.77 | 0.42 | 0.8583 |
| Decision Tree | 0.7084 | 0.17 | 0.47 | 0.25 | 0.6282 |
| Gradient Boosting | 0.7726 | 0.26 | 0.71 | 0.38 | 0.8269 |

## Comparison Summary

Logistic Regression and Linear SVM produced very similar results in the tested configurations.

Logistic Regression achieved a **ROC-AUC of 0.8584** and **77% recall for Class 1**. Linear SVM achieved a ROC-AUC of **0.8583** with the same Class 1 recall.

The tree-based models produced lower Class 1 recall and F1-score in the configurations tested.

Based on the tested results, Logistic Regression is the recommended model for production because it provided strong Class 1 recall and ROC-AUC while remaining relatively simple and computationally efficient.


# Final Model Selection

Based on the tested configurations, Logistic Regression and Linear SVM produced very similar results.

Logistic Regression achieved an accuracy of **78.27%**, Class 1 recall of **0.77**, Class 1 F1-score of **0.42**, and ROC-AUC of **0.8584**.

Linear SVM achieved an accuracy of **78.29%**, Class 1 recall of **0.77**, Class 1 F1-score of **0.42**, and ROC-AUC of **0.8583**.

For this project, **Logistic Regression is recommended as the candidate model for further validation and potential deployment** because it provides comparable predictive performance while offering a simpler and more interpretable classification approach.

Note that Class 1 precision is about 0.29, so many customers flagged as likely to transact will not do so. The bank should account for this in how predictions are used, and validate on unseen business data before deployment.


# Challenges Faced & Solutions

| Challenge | Technique used | Reason |
|---|---|---|
| **Class imbalance** (only ~10% positive) | Random under-sampling on the **training set only** | Balances the classes so models learn to detect Class 1. It is fast on 200,000 rows and adds no synthetic data. The test set keeps its original distribution for a realistic evaluation. |
| **Accuracy is misleading** | Precision, Recall, F1-score, ROC-AUC and Confusion Matrix | A model that predicts "no transaction" for everyone would already reach about 90% accuracy, so accuracy alone says little about Class 1. |
| **Different feature scales** | `StandardScaler`, fitted on the training data only and then applied to the test data | Logistic Regression and Linear SVM are sensitive to feature scale. Fitting on training data only avoids data leakage. Tree-based models do not need scaling, so they use unscaled data. |
| **Outliers** | Inspected with the IQR method, not removed | Features are anonymized, so extreme values cannot be judged as errors. Removing them could discard real customer behaviour. |
| **Large dataset** (200,000 rows, 200 features) | Controlled model settings (e.g. 100 trees, limited depth) | Keeps training time manageable while still allowing a fair comparison of models. |
| **Choosing a model** | Six models compared on the same split and metrics | Covers linear, tree-based and boosting approaches, so the recommendation is based on evidence rather than a single model. |
| **Anonymized features** | Focus on prediction, not feature interpretation | Feature names are not provided, so individual features cannot be explained. |


# Conclusion

A machine learning classification system was developed to predict future customer transactions.

The data was analyzed, class imbalance was handled using random under-sampling on the training data, and multiple classification models were evaluated.

Among the tested models, **Logistic Regression** achieved a ROC-AUC of **0.8584** and Class 1 recall of **0.77**.

Based on the tested configurations, **Logistic Regression is recommended as the best model for production**, with validation on unseen data before deployment.


# Future Improvements

* Apply advanced techniques such as **SMOTE** for handling class imbalance.
* Perform **hyperparameter tuning** to improve model performance.
* Use **cross-validation** for more reliable model evaluation.
* Apply **feature selection** to identify useful features.
* Add **model explainability** to understand predictions.
* Deploy the final validated model as a **web application or API**.
* Monitor model performance using new customer data.
